In [5]:
import pandas as pd
df = pd.read_csv("planets.csv")
df.head()

,rowid,pl_hostname,pl_letter,pl_discmethod,pl_pnum,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,...,st_masserr1,st_masserr2,st_masslim,st_massblend,st_rad,st_raderr1,st_raderr2,st_radlim,st_radblend,rowupdate
0,1,11 Com,b,Radial Velocity,1,326.03,0.32,-0.32,0.0,1.290,...,0.30,-0.30,0.0,0.0,19.00,2.00,-2.00,0.0,0.0,2014-05-14
1,2,11 UMi,b,Radial Velocity,1,516.22,3.25,-3.25,0.0,1.540,...,0.25,-0.25,0.0,0.0,24.08,1.84,-1.84,0.0,0.0,2014-05-14
2,3,14 And,b,Radial Velocity,1,185.84,0.23,-0.23,0.0,0.830,...,0.10,-0.20,0.0,0.0,11.00,1.00,-1.00,0.0,0.0,2014-05-14
3,4,14 Her,b,Radial Velocity,1,1773.40,2.50,-2.50,0.0,2.770,...,0.05,-0.05,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2014-05-14
4,5,16 Cyg B,b,Radial Velocity,1,798.50,1.00,-1.00,0.0,1.681,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2015-09-10


In [2]:
print("Valores nulos por columna:")
print(df.isnull().sum())

Valores nulos por columna:
rowid              0
pl_hostname        0
pl_letter          0
pl_discmethod      0
pl_pnum            0
                ... 
st_raderr1       417
st_raderr2       494
st_radlim        358
st_radblend      187
rowupdate          0
Length: 67, dtype: int64


In [6]:
# Limpieza general para planets.csv 
import re
import numpy as np

# trabajar sobre una copia
df_clean = df.copy()
print("Forma inicial:", df_clean.shape)

# 1) Duplicados
antes = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print("Duplicados eliminados:", antes - len(df_clean))

# 2) Eliminar columnas con muchos nulos (>80%)
pct_nulos = df_clean.isnull().mean()
drop_many = pct_nulos[pct_nulos > 0.80].index.tolist()
print("Columnas >80% nulas (se eliminan):", drop_many)
df_clean.drop(columns=drop_many, inplace=True, errors='ignore')
df_clean.head(5)



Forma inicial: (3372, 67)
Duplicados eliminados: 0
Columnas >80% nulas (se eliminan): ['pl_orbincl', 'pl_orbinclerr1', 'pl_orbinclerr2', 'pl_orbincllim', 'pl_dens', 'pl_denserr1', 'pl_denserr2', 'pl_denslim', 'st_optmagerr']


,rowid,pl_hostname,pl_letter,pl_discmethod,pl_pnum,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,...,st_masserr1,st_masserr2,st_masslim,st_massblend,st_rad,st_raderr1,st_raderr2,st_radlim,st_radblend,rowupdate
0,1,11 Com,b,Radial Velocity,1,326.03,0.32,-0.32,0.0,1.290,...,0.30,-0.30,0.0,0.0,19.00,2.00,-2.00,0.0,0.0,2014-05-14
1,2,11 UMi,b,Radial Velocity,1,516.22,3.25,-3.25,0.0,1.540,...,0.25,-0.25,0.0,0.0,24.08,1.84,-1.84,0.0,0.0,2014-05-14
2,3,14 And,b,Radial Velocity,1,185.84,0.23,-0.23,0.0,0.830,...,0.10,-0.20,0.0,0.0,11.00,1.00,-1.00,0.0,0.0,2014-05-14
3,4,14 Her,b,Radial Velocity,1,1773.40,2.50,-2.50,0.0,2.770,...,0.05,-0.05,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2014-05-14
4,5,16 Cyg B,b,Radial Velocity,1,798.50,1.00,-1.00,0.0,1.681,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2015-09-10


In [7]:
print("Valores nulos por columna:")
print(df_clean.isnull().sum())

Valores nulos por columna:
rowid                 0
pl_hostname           0
pl_letter             0
pl_discmethod         0
pl_pnum               0
pl_orbper            67
pl_orbpererr1       192
pl_orbpererr2       192
pl_orbperlim         68
pl_orbsmax         1475
pl_orbsmaxerr1     2352
pl_orbsmaxerr2     2353
pl_orbsmaxlim      1480
pl_orbeccen        2398
pl_orbeccenerr1    2671
pl_orbeccenerr2    2671
pl_orbeccenlim     2401
pl_bmassj          2198
pl_bmassjerr1      2385
pl_bmassjerr2      2385
pl_bmassjlim       2198
pl_bmassprov       2198
pl_radj             678
pl_radjerr1         716
pl_radjerr2         716
pl_radjlim          678
pl_ttvflag            0
pl_kepflag            0
pl_k2flag             0
pl_nnotes             0
ra_str                0
ra                    0
dec_str               0
dec                   0
st_dist            1076
st_disterr1        1185
st_disterr2        1185
st_distlim         1079
st_optmag           121
st_optmaglim        121
st_optmagblen

In [8]:
# Estrategia específica para exoplanetas
def rellenar_exoplanetas(df_clean):
    # 1. Agrupar por método de detección (lo más importante)
    for col in df_clean.select_dtypes(include=['float64', 'int64']).columns:
        df_clean[col] = df_clean.groupby('pl_discmethod')[col].transform(
            lambda x: x.fillna(x.median())
        )
    
    # 2. Los que aún quedan NaN, dejarlos así (datos insuficientes)
    return df_clean

In [13]:
print(df_clean.isnull().sum())


df.head(5)
df_clean.head(5)

rowid                 0
pl_hostname           0
pl_letter             0
pl_discmethod         0
pl_pnum               0
pl_orbper            67
pl_orbpererr1       192
pl_orbpererr2       192
pl_orbperlim         68
pl_orbsmax         1475
pl_orbsmaxerr1     2352
pl_orbsmaxerr2     2353
pl_orbsmaxlim      1480
pl_orbeccen        2398
pl_orbeccenerr1    2671
pl_orbeccenerr2    2671
pl_orbeccenlim     2401
pl_bmassj          2198
pl_bmassjerr1      2385
pl_bmassjerr2      2385
pl_bmassjlim       2198
pl_bmassprov       2198
pl_radj             678
pl_radjerr1         716
pl_radjerr2         716
pl_radjlim          678
pl_ttvflag            0
pl_kepflag            0
pl_k2flag             0
pl_nnotes             0
ra_str                0
ra                    0
dec_str               0
dec                   0
st_dist            1076
st_disterr1        1185
st_disterr2        1185
st_distlim         1079
st_optmag           121
st_optmaglim        121
st_optmagblend      121
st_optband      

,rowid,pl_hostname,pl_letter,pl_discmethod,pl_pnum,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,...,st_masserr1,st_masserr2,st_masslim,st_massblend,st_rad,st_raderr1,st_raderr2,st_radlim,st_radblend,rowupdate
0,1,11 Com,b,Radial Velocity,1,326.03,0.32,-0.32,0.0,1.290,...,0.30,-0.30,0.0,0.0,19.00,2.00,-2.00,0.0,0.0,2014-05-14
1,2,11 UMi,b,Radial Velocity,1,516.22,3.25,-3.25,0.0,1.540,...,0.25,-0.25,0.0,0.0,24.08,1.84,-1.84,0.0,0.0,2014-05-14
2,3,14 And,b,Radial Velocity,1,185.84,0.23,-0.23,0.0,0.830,...,0.10,-0.20,0.0,0.0,11.00,1.00,-1.00,0.0,0.0,2014-05-14
3,4,14 Her,b,Radial Velocity,1,1773.40,2.50,-2.50,0.0,2.770,...,0.05,-0.05,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2014-05-14
4,5,16 Cyg B,b,Radial Velocity,1,798.50,1.00,-1.00,0.0,1.681,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2015-09-10


In [ ]:

# Crear columna de densidad
from sklearn.preprocessing import MinMaxScaler

# Crear columna de densidad: masa / (radio³)
df_clean['pl_density'] = df_clean['pl_bmassj'] / (df_clean['pl_radj'] ** 3)

# Normalizar densidad (0-1)
scaler = MinMaxScaler()
df_clean['pl_density_norm'] = scaler.fit_transform(df_clean[['pl_density']])

# Clasificar por tipo de planeta
df_clean['planet_type'] = pd.cut(
    df_clean['pl_density'],
    bins=[0, 0.1, 0.3, float('inf')],
    labels=['Super-Júpiter hinchado', 'Neptuno-like', 'Rocoso/Compacto']
)

print(" Columnas creadas: pl_density, pl_density_norm, planet_type")
print("\n Resumen:")
print(f"   Valores de densidad: {df_clean['pl_density'].notna().sum()} válidos, {df_clean['pl_density'].isna().sum()} NaN")
print(f"\n Distribución de tipos:")
print(df_clean['planet_type'].value_counts(dropna=True))
print(f"\n Primeros 5 registros:")
print(df_clean[['pl_hostname', 'pl_letter', 'pl_bmassj', 'pl_radj', 'pl_density', 'planet_type']].head())


 Columnas creadas: pl_density, pl_density_norm, planet_type

 Resumen:
   Valores de densidad: 509 válidos, 2863 NaN

 Distribución de tipos:
planet_type
Rocoso/Compacto           430
Neptuno-like               67
Super-Júpiter hinchado     12
Name: count, dtype: int64

 Primeros 5 registros:
  pl_hostname pl_letter  pl_bmassj  pl_radj  pl_density planet_type
0      11 Com         b      19.40      NaN         NaN         NaN
1      11 UMi         b      10.50      NaN         NaN         NaN
2      14 And         b       4.80      NaN         NaN         NaN
3      14 Her         b       4.64      NaN         NaN         NaN
4    16 Cyg B         b       1.68      NaN         NaN         NaN


Delínea 60: Calcula la densidad de cada planeta

Fórmula: Masa ÷ (Radio³)
Resultado = qué tan compacto es el planeta
Línea 63-65: Normaliza los valores de densidad a rango 0-1

Convierte valores grandes (como 2644) a un rango manejable
Útil para comparaciones y visualización
Línea 67-73: Clasifica cada planeta en 3 categorías según densidad

Densidad > 0.3 =  Rocoso/Compacto
Densidad 0.1-0.3 =  Neptuno-like
Densidad < 0.1 =  Super-Júpiter hinchado
Línea 75-82: Imprime resumen y estadísticas

Cuántos planetas tienen datos válidos
Distribución de tipos
Primeros 5 ejemplos

🪐 Las 3 categorías de planetas explicadas:
1️⃣ 🪨 ROCOSO/COMPACTO (Densidad > 0.3)
Composición: Principalmente roca y metal
Características:
Muy densa (comprimida)
Pequeña pero pesada
Similar a Tierra, Venus, Marte
Ejemplo: Planeta de 5 masas de Júpiter en 2 radios de Júpiter
Densidad = 5 / 2³ = 0.625 ✓ Rocoso
Atmósfera: Poca o ninguna
2️⃣ 🌍 NEPTUNO-LIKE (Densidad 0.1-0.3)
Composición: Mezcla de roca y gases (Hielo + gases)
Características:
Intermedia entre rocoso y gaseoso
"Mini-Neptuno" o "Super-Tierra con gases"
Más grande que Earth pero más compacto que Júpiter
Ejemplo: Planeta de 2 masas de Júpiter en 2 radios de Júpiter
Densidad = 2 / 2³ = 0.25 ✓ Neptuno-like
Atmósfera: Moderada, con muchos gases
3️⃣ 🪨 SUPER-JÚPITER HINCHADO (Densidad < 0.1)
Composición: Principalmente gases (hidrógeno y helio)
Características:
Muy poco denso (muy esponjoso)
Grande pero muy ligero
Absorbe mucho calor de su estrella → se hincha
Ejemplo: Planeta de 1 masa de Júpiter en 2 radios de Júpiter
Densidad = 1 / 2³ = 0.125 ✓ Super-Júpiter hinchado
Atmósfera: TODA ES atmósfera, sin superficie sólida